   # Price forecast: rules

   - **Target:** hourly Victorian price (price_mwh, $/MWh).
   - **Horizons:** 72 h (main), 96 h and 120 h ahead.
   - **Golden rule:** a forecast made at time T uses only information known at T.
   - **Baseline:** price at the same hour one week earlier (168 h).
   - **Stand-ins:** actual weather and actual demand for the target hour stand in for forecasts of them.
   - **Testing:** walk-forward only. Train on the past, test on the future, never shuffle.

In [2]:
   from pathlib import Path
   import pandas as pd

   PROCESSED = Path("../data/processed")
   master = pd.read_csv(PROCESSED / "master_hourly.csv", parse_dates=["hour_end"])
   master = master.set_index("hour_end")

   TARGET = "price_mwh"
   HORIZONS = [72, 96, 120]    # hours ahead
   MAIN_HORIZON = 72
   BASELINE_LAG = 168          # same hour last week

   print("Rows:", len(master), "| from", master.index.min(), "to", master.index.max())
   print("Target:", TARGET, "| horizons (hours):", HORIZONS)

   def allowed_lags(horizon, candidates=(24, 48, 72, 96, 120, 144, 168, 336)):
       """Price lags (in hours) that are old enough to be known at forecast time."""
       return [lag for lag in candidates if lag >= horizon]

   for h in HORIZONS:
       print(f"{h} hours ahead -> allowed price lags (hours):", allowed_lags(h))

   target_time = pd.Timestamp("2026-01-15 18:00")
   print("Example: forecasting the price for", target_time)
   for h in HORIZONS:
       made_at = target_time - pd.Timedelta(hours=h)
       print(f"  {h} h ahead: forecast made at {made_at}. Latest price it may use: {made_at}")

Rows: 15911 | from 2024-12-01 01:00:00 to 2026-09-24 23:00:00
Target: price_mwh | horizons (hours): [72, 96, 120]
72 hours ahead -> allowed price lags (hours): [72, 96, 120, 144, 168, 336]
96 hours ahead -> allowed price lags (hours): [96, 120, 144, 168, 336]
120 hours ahead -> allowed price lags (hours): [120, 144, 168, 336]
Example: forecasting the price for 2026-01-15 18:00:00
  72 h ahead: forecast made at 2026-01-12 18:00:00. Latest price it may use: 2026-01-12 18:00:00
  96 h ahead: forecast made at 2026-01-11 18:00:00. Latest price it may use: 2026-01-11 18:00:00
  120 h ahead: forecast made at 2026-01-10 18:00:00. Latest price it may use: 2026-01-10 18:00:00


In [3]:
import numpy as np

RESULTS = Path("../results")
RESULTS.mkdir(exist_ok=True)
TEST_START = "2025-10-01"

# Baseline: the forecast for each hour is the actual price 168 hours (one week) earlier
baseline = master[TARGET].shift(BASELINE_LAG)
results = pd.DataFrame({"actual": master[TARGET], "baseline": baseline}).dropna()
test = results[results.index >= TEST_START]

def mae(actual, forecast):
    return float(np.mean(np.abs(actual - forecast)))

def rmse(actual, forecast):
    return float(np.sqrt(np.mean((actual - forecast) ** 2)))

print("Test hours:", len(test), "from", test.index.min(), "to", test.index.max())
print("Baseline MAE : $%.2f/MWh" % mae(test.actual, test.baseline))
print("Baseline RMSE: $%.2f/MWh" % rmse(test.actual, test.baseline))
print("Average actual price in the test period: $%.2f/MWh" % test.actual.mean())

results.to_csv(RESULTS / "baseline_forecast.csv")

Test hours: 8616 from 2025-10-01 00:00:00 to 2026-09-24 23:00:00
Baseline MAE : $53.78/MWh
Baseline RMSE: $91.00/MWh
Average actual price in the test period: $48.21/MWh
